# Training

Project repository: [josepsuy/LPR-GNN](https://github.com/josepsuy/LPR-GNN) (currently private; owner-granted access required). See [README.md](README.md) for installation and [CITATION.cff](CITATION.cff) for the preferred scientific citation.

Training uses the original supervised graph loss and augmentation. Validation runs the same dynamic-graph scheduler as makespan_predictor on complete original RCPSP instances. The checkpoint with minimum mean validation makespan is selected; only afterwards is test evaluated. No validation/test graph files are needed. All settings are in config.yaml.

Raw instances are not bundled. Before running this notebook, follow [DATASETS.md](DATASETS.md) for download sources, placement and frozen-split requirements.


In [ ]:
from pathlib import Path
from src.trainer import load_config, train
from src.instance_evaluation import load_instance_splits
PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'
config = load_config(CONFIG_PATH)
config['_project_root'] = str(PROJECT_ROOT)
config['data']['train_dir'] = str((PROJECT_ROOT / config['data']['train_dir']).resolve())
splits = load_instance_splits(config)
print({split: len(paths) for split, paths in splits.items()})
print('Selection: minimum validation mean makespan. Test only after training.')


## Train
Fresh training is the default. To resume, use a makespan-selected checkpoint with the same split manifest.
The PNG progress dashboard is refreshed after each completed epoch when training.plot_history is true.


In [ ]:
train(CONFIG_PATH)


## Training evolution
Total/BCE/top-1 loss, mean validation makespan and best epoch, training top-1/accuracy/F1, and time per epoch.
This cell only reads `history.csv`: rerun it without training or evaluating the model. It does not show test curves because test evaluation runs only after training.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import yaml
from src.training_plots import plot_training_history

# Independent of the training cell: reload the output path from the YAML.
plot_root = Path.cwd().resolve()
plot_config = yaml.safe_load((plot_root / 'config.yaml').read_text(encoding='utf-8'))
out = plot_root / plot_config['output_dir']
figure = plot_training_history(out / 'history.csv', out / 'training_evolution.png')
if figure is None:
    print('No completed epochs in history.csv yet.')
else:
    plt.show()
    plt.close(figure)
    print('Plot:', out / 'training_evolution.png')


## Selected checkpoint: final validation and test
Results are generated automatically after training. No additional test execution here.


In [ ]:
print((out / 'validation_test_makespans.tex').read_text(encoding='utf-8'))
